# Process and View Results From Multiple Longstrips

1. Point to proper saved / cached .hdf5 file
2. Verify paths.
3. then run With:
    `panel serve .\pnviewer_multistrip_results.ipynb --autoreload --dev --show`

In [ ]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import naatos_oct_tools.panelui.multistrip as multistrip
import naatos_oct_tools.panelui.singlestrip as singlestrip

In [ ]:
# # Magics to autoreload submodules when they are modified
# %load_ext autoreload
# %autoreload 2

In [ ]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

In [ ]:
#%% Filter The Tests To View
# dfmasks = [
#     (dftests['Test date']=='2025-05-08') | (dftests['Test date']=='2025-05-12'),
#     ~dftests['Batch'].str.startswith('Valves')
# ]
dfmasks = [
    (dftests['Test date']>='2025-06-05'),
    #(dftests['Test date']=='2025-06-05'),
    #(dftests['Test date']=='2025-06-11') | (dftests['Test date']=='2025-06-12'),
]


dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

In [ ]:
#%% Cleanup the experiment index to make it useful for us
# excel formula to extract Wax (mg/mm) does not really import right
# we'll recalculate it
idx_nowax_data = dffilt['Wax (mg/mm)'].isna();
def getWaxMaxxFromStripname(value):
    #
    if(type(value)==str):
        #return 'string';
        space_in_string = value.find(' ');
        if(space_in_string>0):
            return float(value[space_in_string+1:])
        else:
            return 0.0;
    else:
        return -1.0;
dffilt.loc[idx_nowax_data,'Wax (mg/mm)'] = dffilt[idx_nowax_data]['Strip'].apply(getWaxMaxxFromStripname)


In [ ]:
dffilt

In [ ]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

folder_temp = Path(r'D:\TEMP')

# Load data

In [ ]:
#df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]
#dfloaded = pd.read_hdf(folder_temp/'oct_results_summary.hdf',key='table');
#dfloaded = pd.read_hdf(folder_temp/'oct_results_summary_2025_06_10.hdf',key='table');
#dfloaded = pd.read_hdf(folder_temp/'oct_results_summary_2025_06_16_newbatch.hdf',key='table');
#dfloaded = pd.read_hdf(folder_temp/'oct_results_summary_2025_07_23_allbatches.hdf',key='table');
dfloaded = pd.read_hdf(folder_temp/'oct_results_summary_2025_07_25b_allbatches.hdf',key='table');

In [ ]:
dffilt

In [ ]:
# strip down columns we will not use below
#df = df[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]];
# join info from spreadsheet
dfloaded = pd.merge(dfloaded.reset_index(),dffilt,left_on='level_0',right_on='Test name');
dfloaded

In [ ]:
# sanitize data, replace wax mass NaNs with zero
dfloaded['Wax (mg/mm)'] = dfloaded['Wax (mg/mm)'].astype(float)

In [ ]:
# further filter
if False:
    batches = [
        'July11WaxrobotStrips',
        'July17n18Oldstriper',
    ]
    dffilt = dffilt.query('Batch==@batches')

In [ ]:
dffilt.columns

In [ ]:
dffilt['Batch'].unique()

# Test Panel

In [ ]:
pn.extension("tabulator")
import param

In [ ]:
import naatos_oct_tools.panelui.metricdefs as metricdefs
metrics = metricdefs.metrics;


In [ ]:
metrics


In [ ]:
OCTMultiStripMetricViewer = multistrip.OCTMultiStripMetricViewer;
OCTSingleStripMetricViewer = singlestrip.OCTSingleStripMetricViewer;

#import naatos_oct_tools.panelui.metricdefs as metricdefs
#metricdefs.m

In [ ]:
# obj2 = OCTMultiStripMetricViewer(
#     #['GHL_pyapp_20250512T0947', 'GHL_pyapp_20250512T0951', 'GHL_pyapp_20250512T0955', 'GHL_pyapp_20250512T0959', 'GHL_pyapp_20250512T1002', 'GHL_pyapp_20250512T1117', 'GHL_pyapp_20250512T1121', 'GHL_pyapp_20250512T1125', 'GHL_pyapp_20250512T1130', 'GHL_pyapp_20250512T1133', 'GHL_pyapp_20250512T1137', 'GHL_pyapp_20250512T1140', 'GHL_pyapp_20250512T1144', 'GHL_pyapp_20250512T1149', 'GHL_pyapp_20250512T1152', 'GHL_pyapp_20250512T1158', 'GHL_pyapp_20250512T1201', 'GHL_pyapp_20250512T1204', 'GHL_pyapp_20250512T1208', 'GHL_pyapp_20250512T1211', 'GHL_pyapp_20250512T1215', 'GHL_pyapp_20250512T1218', 'GHL_pyapp_20250512T1222', 'GHL_pyapp_20250512T1225', 'GHL_pyapp_20250512T1230', 'GHL_pyapp_20250512T1238', 'GHL_pyapp_20250512T1241', 'GHL_pyapp_20250512T1245', 'GHL_pyapp_20250512T1248', 'GHL_pyapp_20250512T1251', 'GHL_pyapp_20250512T1254', 'GHL_pyapp_20250512T1258', 'GHL_pyapp_20250512T1303'],
#     ['GHL_pyapp_20250611T1704','GHL_pyapp_20250611T1709'],
#     dfloaded
# );

In [ ]:
#fig = obj2._mkfigure('scatterVsLength',metrics);
#fig = fig[0].object
#fig.show(renderer='browser')

In [ ]:
# test standalone single strip viewer
# obj3 = OCTSingleStripMetricViewer(
#     #'GHL_pyapp_20250512T0947'
#     #'GHL_pyapp_20250611T1709'
#     #'GHL_pyapp_20250612T0752',
#     'GHL_pyapp_20250612T0807',
# );
#fig = obj3._mkfig()
#fig.show(renderer='browser')

In [ ]:
#fig.data[0].name

In [ ]:
class OCTStripExplorer(pn.viewable.Viewer):
    # follow https://panel.holoviz.org/tutorials/intermediate/interactivity.html
    # from   the "with pn.rx" class

    data_table = param.DataFrame(doc="List Of Strips From Excel File")
    page_size = param.Integer(default=10, doc="Number of rows per page.", bounds=(1, None))

    # c_def_cols = ["unit","expname","run","TimeBeg","status","text"];
    # columns = param.ListSelector(
    #     default=["p_name", "t_state", "t_county", "p_year", "t_manu", "p_cap"]
    # )
    
    filtered_data = param.Parameter()

    number_of_rows = param.Parameter()

    # button1 = param.Action(label='Reset');
    # button2 = param.Action(label='Update');
    # button1 = param.Action(label='Refine');
    # button2 = param.Action(label='Reset');
    actionbutton = param.Action(lambda x: x.param.trigger('actionbutton'), label='Update plot!');
    refinebutton = param.Action(lambda x: x.param.trigger('refinebutton'), label='Refine choices');
    resetbutton = param.Action(lambda x: x.param.trigger('resetbutton'), label='Reset choices');
    # button2 = param.Action(lambda x: x.param.trigger('button2'), label='Start [spacebar]');
    # button3 = param.Action(lambda x: x.param.trigger('button3'), label='Insert[spacebar]');
    # button4 = param.Action(lambda x: x.param.trigger('button4'), label='Stop');
    wax_mass_range = param.Range(step=0.001);


    c_filters = {
        "Batch":param.ListSelector,
        "Test name":param.ListSelector,
    };

    multi_graph_obj = pn.pane.Str('multi graph origin');
    single_graph_obj = pn.pane.Str('single graph origin');

    @param.depends('refinebutton', watch=True)
    def _refine_button(self):
        print('Refine choices');
        self._update_or_add_params(filtered=True);
    
    def _update_or_add_params(self,filtered=False):
        df : pd.DataFrame;
        if(not filtered):
            df = self.data_table;
        else:
            df = self.filtered_data.rx.value;
        
        print('Dataframe DF type:',type(df));

        for col,paramobj in self.c_filters.items():
            print('Working on',col,filtered);
            #print(self.data_table)
            # unique items

            items = df[col].unique().tolist();
    
            #print( 'Decision Tree, type(paramobj)' , str(type(paramobj)) );
            if(paramobj == param.ListSelector):
                #print('LIST SELECTOR:',paramobj.name)
                if col not in self.param:
                    # add brand new parameter
                    #print('Here1')
                    newParam = paramobj(default=sorted(items),objects=sorted(items));
                    #newParam = pn.Param(paramobj(default=sorted(items),objects=sorted(items)), widgets={'height':100});
                    print('Adding param',col,'df.shape',df.shape)
                    self.param.add_parameter(col,newParam);
                else:
                    # update parameter
                    print('Updating param',col,'df.shape',df.shape)

                    # reset available options
                    self.param[col].objects = items;
                    # reset selections (to all)
                    setattr(self,col,items);
            elif(paramobj == param.DateRange):
                print('DateRange (not implemented)');
                
            else:
                print('OTHER');

        # Wax Valve Range Slider
        if( df['Wax (mg/mm)'].min() == df['Wax (mg/mm)'].max()):
            self.param.wax_mass_range.bounds = ( np.nan , np.nan );
        else:
            self.param.wax_mass_range.bounds = (df['Wax (mg/mm)'].min(),df['Wax (mg/mm)'].max());
        self.wax_mass_range = (df['Wax (mg/mm)'].min(),df['Wax (mg/mm)'].max())
        print('HereFinal')

    def _filter(self):
        dfrx = self.param.data_table.rx();

        # p_year_min = self.param.year.rx().rx.pipe(lambda x: x[0])
        # p_year_max = self.param.year.rx().rx.pipe(lambda x: x[1])
        # p_cap_min = self.param.capacity.rx().rx.pipe(lambda x: x[0])
        # p_cap_max = self.param.capacity.rx().rx.pipe(lambda x: x[1])

        # self.filtered_data = dfrx[
        #     dfrx.p_year.between(p_year_min, p_year_max)
        #     & dfrx.p_cap.between(p_cap_min, p_cap_max)
        # ][self.param.columns]
        
        # filter the columns
        # masks = [];
        # for col in self.c_filters.keys():

        #     pfilt = self.param[col].rx().rx;
        #     print(col,pfilt);
        #     masks.append(
        #         dfrx[col].isin(pfilt)
        #     )

        # # masks = [
        # #     dfrx
        # # ]
        # print(masks);
        # for col in self.c_filters.keys():
        #     dfrx = dfrx[dfrx.isin(self.param[col].rx())];

        #self.filtered_data = dfrx[dfrx['Batch'].isin(self.param['Batch'].rx())];
        # for col in self.c_filters.keys():
        #     print('filter col',col);
        #     dfrx = dfrx[dfrx.isin(self.param[col].rx())];
        # masks = [];
        # pfiltrxlist = [];
        # for col in self.c_filters.keys():
        #     print('filter col',col);
        #     pfiltrx = self.param[col].rx();
        #     pfiltrxlist.append(pfiltrx);
        #     masks.append( dfrx[col].isin(pfiltrx) );
        # #print(np.all(np.vstack(masks),axis=1))
        # dfrx = dfrx[ np.logical_and.reduce(masks,axis=0) ];
        #print(self.wax_mass_range)
        p_wax_mass_range_min = self.param.wax_mass_range.rx().rx.pipe(lambda x: x[0]);
        p_wax_mass_range_max = self.param.wax_mass_range.rx().rx.pipe(lambda x: x[1]);
        dfrx = dfrx[
            dfrx['Batch'].isin(self.param['Batch'].rx()) &
            dfrx['Test name'].isin(self.param['Test name'].rx()) &
            dfrx['Wax (mg/mm)'].between( p_wax_mass_range_min , p_wax_mass_range_max )
        ];

        self.filtered_data = dfrx;

        self.number_of_rows = pn.rx("Scans: {len_df}").format(len_df=pn.rx(len)(dfrx))


    def __init__(self, data_table, **params):
        super().__init__(**params)
        self.data_table = data_table;
        #self.param.columns.objects = self.data_table.columns.to_list()
        self._update_or_add_params();
        self._reset_choices();
        self._filter();
    
        #self.wax_mass_range.param
        #self.param.wax_mass_range.bounds = (self.filtered_data.)

    @param.depends('resetbutton', watch=True)
    def _reset_choices(self):
        dfrx = self.param.data_table;
        self.filtered_data = dfrx;
        self._update_or_add_params(filtered=True);
    
    # @param.depends('refinebutton', watch=True)
    # def _refine_choices(self):
    #     print('Refine choices');
    #     #self._update_or_add_params(filtered=True);

    def click_handling_2_multigraph(self,event):
        print("click_handling_2_multigraph datatype:{:s} data:{:s}", type(event), str(event));
    
        if not event and (self.multi_graph_obj.plotpane.object is None):
            print( "No point clicked" );
            return;
            pass;
        try:
            point = event["points"][0]
            curvenumber = point['curveNumber'];
            index = point['pointIndex']
            x = point['x']
            y = point['y']
            if(type(x) is not str):
                # x was a number
                # so.... find the name buried inside the plotly plot legend
                scanname = self.multi_graph_obj.plotpane.object.data[curvenumber].name;
            else:
                scanname = x;
        except Exception as ex:
            print( f"You clicked the Plotly Chart! I could not determine the point: {ex}" )
            return;
        
        print( f"**You clicked point index {index} at ({x}, {y}) on curve ({scanname}) in the Plotly Chart!**" )

        # instance a new single graph obj
        self.single_graph_obj = OCTSingleStripMetricViewer(scanname);
        self.tabs[2] = ('3-SinglestripMetrics',pn.Column(self.single_graph_obj));

    @param.depends('actionbutton', watch=True)
    def _update_plot(self):
        print('Update plot!');
        df = self.filtered_data.rx.value;
        #print(self.filtered_data['Test name'].unique().tolist())
        self.multi_graph_obj = OCTMultiStripMetricViewer(df['Test name'].unique().tolist(),dfloaded=dfloaded);
        print(self.multi_graph_obj.plotpane)
        iclickbind = pn.bind(self.click_handling_2_multigraph, self.multi_graph_obj.plotpane.param.click_data);
        self.tabs[1] = ('2-MultistripMetrics',pn.Column(iclickbind,self.multi_graph_obj));
        #print('Update plot! 2');

    def __panel__(self):
        stylesheet = """
        .tabulator-cell {
            font-size: 9px;
        }
        """
        main_panel = pn.Column(
            # pn.Row(
            #     pn.widgets.MultiChoice.from_param(self.param.columns, width=400),
            #     pn.Column(self.param.year, self.param.capacity),
            # ),
            pn.Row(
                pn.Column(
                    self.param.actionbutton,self.param.refinebutton,self.param.resetbutton
                ),
                *[self.param[x] for x in self.c_filters.keys()],
                self.param.wax_mass_range,
            ),
            self.number_of_rows,
            pn.Row(
                pn.widgets.Tabulator(self.filtered_data, page_size=10, pagination="remote",stylesheets=[stylesheet],sizing_mode= 'stretch_width',width=1200)
            ),
            #width_policy=''
        );
        self.tabs = pn.Tabs(
            ('1-Main',main_panel),
            ('2-MultistripMetrics',self.multi_graph_obj),
            ('3-SinglestripMetrics',self.single_graph_obj),
            
            #width=1000
        )
        return self.tabs;
    
obj = OCTStripExplorer(dffilt);
finpn = pn.Column(obj);


In [ ]:
obj.c_filters.keys()

In [ ]:
obj.param['Batch'].rx()

In [ ]:
# class EditableRange(pn.viewable.Viewer):
#     value = param.Range(doc="A numeric range.")
#     width = param.Integer(default=300)

#     def __init__(self, **params):
#         self._start_input = pn.widgets.FloatInput()
#         self._end_input = pn.widgets.FloatInput(align='end')
#         super().__init__(**params)
#         self._layout = pn.Row(self._start_input, self._end_input)
#         self._sync_widgets()

#     def __panel__(self):
#         return self._layout

#     @param.depends('value', 'width', watch=True)
#     def _sync_widgets(self):
#         self._start_input.name = self.name
#         self._start_input.value = self.value[0]
#         self._end_input.value = self.value[1]
#         self._start_input.width = self.width//2
#         self._end_input.width = self.width//2

#     @param.depends('_start_input.value', '_end_input.value', watch=True)
#     def _sync_params(self):
#         self.value = (self._start_input.value, self._end_input.value)

# # range_widget = EditableRange(name='Range', value=(0, 10))

# # finpn = pn.Column(
# #     '#### This is a custom widget',
# #     range_widget
# # );


In [ ]:
# def session_key_func(request):
#     key = request.arguments.get('expname', [def_expname.encode()])[0]+'_'+request.arguments.get('unit', [def_unit.encode()])[0]+'_'+request.arguments.get('run', [def_run.encode()])[0];
#     print('session_key_func',key);
#     return key;

# #pn.extension(template='material', session_key_func=session_key_func)
# pn.extension(session_key_func=session_key_func)

# obj = DataExplorer(data=dffull)
# obj2 = Plotter1App(dffull = dfraw, dfbuilt = dfbuilt);

# def page1():
#     #return obj.servable(location=True)
#     return obj;

# def page_rundetail():
#     #pn.state.location.sync(obj2,['unit']);
#     pn.state.location.sync(obj2, ['expname','unit','run']);
#     #return obj2.servable()
#     return obj2;

def page1():
    #return finpn;
    return OCTStripExplorer(dffilt);

ROUTES = {
    "": page1,
    #"rundetail": page_rundetail
}
# try:
#     serve.stop();
# except Exception as e:
#     print(e)

#serve = pn.serve(ROUTES, port=5006,location=True,verbose=True,admin=True);
page1().servable(target='page1');

In [ ]:
# try:
#     print('Check existing server',str(server))
#     print('Stopping server...')
#     server.stop();
#     del server;
# except:
#     print('Server was not running.')
#     pass;
# server = pn.serve(finpn,show=True)

In [ ]:
# server.stop()